In [109]:
from typing import Literal
import requests
from pathlib import Path
import json
import mimetypes

# APIのエンドポイントURL
API_URL = "http://localhost:8001/extract/text"

# ==========================================
# APIテスト関数の定義
# ==========================================
def test_extract_api(file_path: str, mode: Literal["standard", "deep"] = "standard"):
    path = Path(file_path)

    # ファイルの存在確認
    if not path.exists():
        print(f"エラー: 指定されたファイルが見つかりません -> {path.absolute()}")
        return

    # MIMEタイプを拡張子から動的に判定
    content_type, _ = mimetypes.guess_type(path)
    # 判定できなかった場合のフォールバック
    if content_type is None:
        content_type = "application/octet-stream"

    # multipart/form-data 形式でファイルを送信
    with open(path, "rb") as file:
        # 動的に取得した content_type を指定する
        files = {"file": (path.name, file, content_type)}
        params = {
            "mode": mode
        }
        
        try:
            # APIへPOSTリクエストを送信
            response = requests.post(API_URL, files=files, params=params)
            if response.status_code == 200:               
                result = response.json()
                return result
                    
            else:
                print(f"エラー発生 (ステータスコード: {response.status_code})")
                # エラー詳細を整形して表示
                print(json.dumps(response.json(), indent=2, ensure_ascii=False))

        except requests.exceptions.ConnectionError:
            print("サーバーに接続できません。")
            print("FastAPIのサーバー (uvicorn) が起動しているか、URLが正しいか確認してください。")

In [ ]:
# --- テスト用ファイルのパスを指定 ---
# .docx形式
TARGET_FILE_PATH = "../data/sample_docx_data/sample_docx_5.docx"
# .xlsx形式
# TARGET_FILE_PATH = "../data/sample_xlsx_data/sample_xlsx_4.xlsx"
# .pptx形式
TARGET_FILE_PATH = "../data/sample_pptx_data/sample_pptx_1.pptx"
# .pdf形式
TARGET_FILE_PATH = "../data/sample_pdf_data/sample_pdf_1.pdf"
# .pdf形式（J-PlatPat特許公報）
TARGET_FILE_PATH = "../data/sample_patent_pdf_data/sample_patent_pdf_1.pdf"
# .jpg形式
TARGET_FILE_PATH = "../data/sample_img_data/sample_img_1.jpg"

# 抽出実行
result = test_extract_api(file_path=TARGET_FILE_PATH, mode="deep")


if result:
    print(f"ファイル名: {result.get('filename')}")
    print(f"ファイル形式: {result.get('file_type')}")
    print("-" * 40, "\n")

    # 抽出されたテキストのプレビュー
    extracted_text = result.get('extracted_text', '')
    print(extracted_text)

ファイル名: sample_img_1.jpg
ファイル形式: jpg
抽出文字数: None 文字
---------------------------------------- 

Supported input formats

Format Description
PDF
DOCX, XLSX, PPTX Default formats in MS Office 2007+, based on Office Open XML
Markdown
AsciiDoc Human-readable, plain-text markup language for structured technical content
LaTeX Scientific document preparation system
HTML, XHTML
CSV
PNG, JPEG, TIFF, BMP, WEBP Image formats
WAV, MP3, M4A, AAC, OGG, FLAC Audio formats (requires asr extra — see Processing audio and video)
MP4, AVI, MOV Video formats — audio track is extracted and transcribed (requires asr extra and ffmpeg)
WebVTT Web Video Text Tracks format for displaying timed text
